In [0]:
df_silver = spark.read.table("he_prod_incen_analytics_dbw_01.silver.fact_encounters")
df_dim_members = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_members")
df_dim_providers = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_providers")
df_dim_facilities = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_facilities")
df_dim_date = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_date")
df_dim_service = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_service_types")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, coalesce, lit, col

df_gold = df_silver.withColumn("encounter_sk", monotonically_increasing_id() + 1)

# Lookups
df_gold = df_gold.join(df_dim_members.select(col("member_sk").alias("gold_member_sk"), "member_id"), on="member_id", how="left").withColumn("gold_member_sk", coalesce(col("gold_member_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_provider_sk"), "provider_id"), on="provider_id", how="left").withColumn("gold_provider_sk", coalesce(col("gold_provider_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_facilities.select(col("facility_sk").alias("gold_facility_sk"), "facility_id"), on="facility_id", how="left").withColumn("gold_facility_sk", coalesce(col("gold_facility_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_date.select(col("date_sk").alias("gold_date_sk"), col("date").alias("encounter_date")), on="encounter_date", how="left").withColumn("gold_date_sk", coalesce(col("gold_date_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_service.select(col("service_type_sk").alias("gold_service_type_sk"), "service_type_id"), on="service_type_id", how="left").withColumn("gold_service_type_sk", coalesce(col("gold_service_type_sk"), lit(-1)))

# Referring provider lookup (alias provider_id to avoid join ambiguity)
df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_referring_provider_sk"), col("provider_id").alias("referring_provider_id")), on="referring_provider_id", how="left").withColumn("gold_referring_provider_sk", coalesce(col("gold_referring_provider_sk"), lit(-1)))

df_gold = df_gold.select(
    "encounter_sk", "encounter_id",
    col("gold_member_sk").alias("member_sk"), col("gold_provider_sk").alias("provider_sk"), 
    col("gold_facility_sk").alias("facility_sk"), col("gold_date_sk").alias("encounter_date_sk"),
    col("gold_service_type_sk").alias("service_type_sk"), col("gold_referring_provider_sk").alias("referring_provider_sk"),
    "encounter_start_time", "encounter_end_time", "duration_minutes", "encounter_type", 
    "diagnosis_count", "procedure_count", "medication_count", "patient_satisfaction", "follow_up_required"
)

display(df_gold.limit(5))

In [0]:
df_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("he_prod_incen_analytics_dbw_01.gold.fact_encounters")
print("✅ Successfully wrote Fact_Encounters to Gold layer.")